In [153]:
#datasets:https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2021-2023
#NHANES August 2021-August 2023

import pandas as pd
import numpy as np

path = "/Users/sherrywang/Desktop/Projects/Dataset"

demo = pd.read_sas(path + "/DEMO_L.XPT")
bmx  = pd.read_sas(path + "/BMX_L.XPT")
bpx  = pd.read_sas(path + "/BPXO_L.XPT")
lab  = pd.read_sas(path + "/GHB_L.XPT")
glu  = pd.read_sas(path + "/GLU_L.XPT")
smq  = pd.read_sas(path + "/SMQ_L.XPT")
paq  = pd.read_sas(path + "/PAQ_L.XPT")
slq  = pd.read_sas(path + "/SLQ_L.XPT")
diq  = pd.read_sas(path + "/DIQ_L.XPT")
inq  = pd.read_sas(path + "/INQ_L.XPT")
dbq  = pd.read_sas(path + "/DBQ_L.XPT")
hiq  = pd.read_sas(path + "/HIQ_L.XPT")
hsq  = pd.read_sas(path + "/HSQ_L.XPT")
huq  = pd.read_sas(path + "/HUQ_L.XPT")
ocq  = pd.read_sas(path + "/OCQ_L.XPT")
alq  = pd.read_sas(path + "/ALQ_L.XPT")
agp  = pd.read_sas(path + "/AGP_L.XPT")


dfs = [demo, bmx, bpx, lab, glu, smq, paq, slq, diq, inq, dbq, hiq, hsq, huq, ocq, alq, agp]

nhanes = dfs[0]
for t in dfs[1:]:
    nhanes = nhanes.merge(t, on="SEQN", how="left")
    
print(nhanes.head())
print(nhanes.shape)

for col in nhanes.columns:
#    print(col)

    cols_map = {
    "SEQN": "id",
    "RIDAGEYR": "age",
    "RIAGENDR": "gender",
    "RIDRETH3": "race",
    "DMDEDUC2": "education",
    "INDFMPIR": "income_poverty_ratio", #A ratio of family income to poverty guidelines.
    
    "BMXWT": "weight_kg",
    "BMXHT": "height_cm",
    "BMXBMI": "bmi",
    
    "BPXOSY1": "sbp",
    "BPXODI1": "dbp",
    
    "LBXGH": "a1c",
    "LBXGLU": "glucose",
    
    "SMQ020": "smoking_current",
    "SMQ040": "smoking_ever",
    
    "PAD820": "physical_activity_freq", #About how long {do you/does SP} do these vigorous leisure-time physical activities each time?
    "SLD012": "sleep_hours",
    
    "DIQ010": "diabetes_self_report",
    "HIQ011": "has_health_insurance",
    "HUQ010": "doctor_visit_12m"
}

nhanes = nhanes[list(cols_map.keys())].rename(columns=cols_map)

type(nhanes)
# Gender
gender_map = {
    1: "Male",
    2: "Female"
}

# Race (NHANES RIDRETH3)
race_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other Race / Multi"
}

# Education
education_map = {
    1: "<9th grade",
    2: "9-11th grade",
    3: "High school/GED",
    4: "Some college/AA",
    5: "College graduate or above"
}

# Smoking
yesno_map = {
    1: "Yes",
    2: "No",
    7: "Refused",
    9: "Don't know"
}

# Diabetes self report
diabetes_map = {
    1: "Yes",
    2: "No",
    3: "Borderline",
    7: "Refused",
    9: "Don't know"
}

# Health insurance
insurance_map = {
    1: "Insured",
    2: "Not insured",
    7: "Refused",
    9: "Don't know"
}

# Doctor visit in last 12 months
visit_map = {
    1: "Yes",
    2: "No",
    7: "Refused",
    9: "Don't know"
}

nhanes['gender'] = nhanes['gender'].map(gender_map)
nhanes['race'] = nhanes['race'].map(race_map)
nhanes['education'] = nhanes['education'].map(education_map)

nhanes['smoking_current'] = nhanes['smoking_current'].map(yesno_map)
nhanes['smoking_ever'] = nhanes['smoking_ever'].map(yesno_map)

nhanes['diabetes_self_report'] = nhanes['diabetes_self_report'].map(diabetes_map)
nhanes['has_health_insurance'] = nhanes['has_health_insurance'].map(insurance_map)
nhanes['doctor_visit_12m'] = nhanes['doctor_visit_12m'].map(visit_map)

#Add insurance type
hiq_cols = ['SEQN'] + [c for c in hiq.columns if str(c).upper().startswith('HIQ032')]
hiq_32 = hiq[hiq_cols].copy()

nhanes = nhanes.merge(
    hiq_32, left_on='id', right_on='SEQN', how='left', suffixes=('', '_hiq')
)
nhanes.drop(columns=['SEQN'], inplace=True)


hiq32_cols_in_df = [c for c in nhanes.columns if str(c).upper().startswith('HIQ032')]
print("HIQ032 columns in nhanes:", hiq32_cols_in_df[:20])

import re
bases = {}
pat = re.compile(r'^(HIQ032[ABCDEFGHI])(?:_.+)?$', re.I)
for c in hiq32_cols_in_df:
    m = pat.match(c)
    if m:
        base = m.group(1).upper()
        s = pd.to_numeric(nhanes[c], errors='coerce')
        # keep the series with more non-nulls if the base already exists
        if base not in bases or s.notna().sum() > bases[base].notna().sum():
            bases[base] = s

hiq32 = pd.DataFrame(bases) if bases else pd.DataFrame(index=nhanes.index)
print("Selected bases & non-null counts:",
      {k:int(v.notna().sum()) for k,v in bases.items()})

# --- 3) flags (multiple-mention format): private=code 1, public=any of {2,3,4,5,6,8,9} ---
has_any = (nhanes['has_health_insurance'] == 'Insured') | (nhanes.get('HIQ011') == 1)

private_flag = hiq32.eq(1).any(axis=1).fillna(False).to_numpy() if not hiq32.empty else np.zeros(len(nhanes), bool)
public_codes = {2,3,4,5,6,8,9}
public_flag  = hiq32.isin(public_codes).any(axis=1).fillna(False).to_numpy() if not hiq32.empty else np.zeros(len(nhanes), bool)


private_only = has_any & private_flag & np.logical_not(public_flag)
public_only  = has_any & np.logical_not(private_flag) & public_flag
both         = has_any & private_flag & public_flag
uninsured    = (nhanes['has_health_insurance'] == 'Not insured') | (nhanes.get('HIQ011') == 2)

nhanes['insurance_type'] = np.select(
    [private_only, public_only, both, uninsured],
    ['Private only', 'Public only', 'Both', 'Uninsured'],
    default='Unknown'
)
hiq_drop_cols = [c for c in nhanes.columns if str(c).upper().startswith("HIQ032")]
nhanes.drop(columns=hiq_drop_cols, inplace=True)

print("\ninsurance_type counts:")
print(nhanes['insurance_type'].value_counts(dropna=False))
nhanes.head()
print(nhanes.shape)
#nhanes.to_csv("/Users/sherrywang/Desktop/Projects/Dataset/nhanes.csv", index=False)


       SEQN  SDDSRVYR  RIDSTATR  RIAGENDR  RIDAGEYR  RIDAGEMN  RIDRETH1  \
0 130378.00     12.00      2.00      1.00     43.00       NaN      5.00   
1 130379.00     12.00      2.00      1.00     66.00       NaN      3.00   
2 130380.00     12.00      2.00      2.00     44.00       NaN      2.00   
3 130381.00     12.00      2.00      2.00      5.00       NaN      5.00   
4 130382.00     12.00      2.00      1.00      2.00       NaN      3.00   

   RIDRETH3  RIDEXMON  RIDEXAGM  ...  ALQ111  ALQ121  ALQ130  ALQ142  ALQ270  \
0      6.00      2.00       NaN  ...     NaN     NaN     NaN     NaN     NaN   
1      3.00      2.00       NaN  ...    1.00    2.00    3.00    0.00     NaN   
2      2.00      1.00       NaN  ...    1.00   10.00    1.00    0.00     NaN   
3      7.00      1.00     71.00  ...     NaN     NaN     NaN     NaN     NaN   
4      3.00      2.00     34.00  ...     NaN     NaN     NaN     NaN     NaN   

   ALQ280  ALQ151  ALQ170  WTPH2YR_y  LBXAGP  
0     NaN     NaN    

In [154]:
import pandas as pd
import numpy as np
from IPython.display import display

# Make a copy of the original dataset
nhanes = pd.read_csv('/Users/sherrywang/Desktop/Data_Mining/Project/data-mining-project-starter/data/processed/nhanes.csv')
nhanes_clean = nhanes.copy()

# Dataframe Info (structure, data types, and missing counts)
nhanes_clean.info(memory_usage='deep')

# Shape, data types, and basic statistics
pd.set_option('display.float_format', '{:.2f}'.format)
print('\nshape:', nhanes_clean.shape)
# A1C normal range: 4.0–5.6%
# Glucose: Normal: 70–99 mg/dL, Prediabetes: 100–125 mg/dL, and Diabetes: ≥126 mg/dL
display(nhanes_clean.dtypes.value_counts())
display(nhanes_clean.describe(include='number'))
display(nhanes_clean.describe(include='object'))

# Check for duplicate unique identifiers
dup_ids = nhanes_clean['id'].duplicated().sum
print(f"\nDuplicated id count: {dup_ids}")

# Missing value percentage sorted
missing_percent = nhanes_clean.isna().mean().sort_values(ascending=False)
print('\n')
print('Missing Percentage by Variable')
display(missing_percent)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11933 entries, 0 to 11932
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      11933 non-null  float64
 1   age                     11933 non-null  float64
 2   gender                  11933 non-null  object 
 3   race                    11933 non-null  object 
 4   education               7783 non-null   object 
 5   income_poverty_ratio    9892 non-null   float64
 6   weight_kg               8754 non-null   float64
 7   height_cm               8499 non-null   float64
 8   bmi                     8471 non-null   float64
 9   sbp                     7517 non-null   float64
 10  dbp                     7517 non-null   float64
 11  a1c                     6715 non-null   float64
 12  glucose                 3672 non-null   float64
 13  smoking_current         8135 non-null   object 
 14  smoking_ever            1190 non-null 

float64    12
object      9
Name: count, dtype: int64

,id,age,income_poverty_ratio,weight_kg,height_cm,bmi,sbp,dbp,a1c,glucose,physical_activity_freq,sleep_hours
count,11933.00,11933.00,9892.00,8754.00,8499.00,8471.00,7517.00,7517.00,6715.00,3672.00,3687.00,8388.00
mean,136344.00,38.32,2.71,70.55,159.66,27.25,119.29,72.75,5.71,107.88,97.57,7.76
std,3444.90,25.60,1.67,30.39,19.86,8.14,18.56,11.90,1.05,32.48,605.42,1.62
min,130378.00,0.00,0.00,2.70,79.10,11.10,61.00,33.00,3.20,59.00,1.00,2.00
25%,133361.00,13.00,1.18,54.20,154.40,21.60,106.00,64.00,5.20,93.00,30.00,7.00
50%,136344.00,37.00,2.50,71.70,163.60,26.40,117.00,72.00,5.50,100.00,45.00,8.00
75%,139327.00,62.00,4.50,89.10,172.10,31.70,130.00,80.00,5.80,109.00,60.00,8.50
max,142310.00,80.00,5.00,248.20,200.70,74.80,232.00,142.00,17.10,561.00,9999.00,14.00


,gender,race,education,smoking_current,smoking_ever,diabetes_self_report,has_health_insurance,doctor_visit_12m,insurance_type
count,11933,11933,7783,8135,1190,11740,11910,6717,11933
unique,2,6,5,4,2,4,4,4,5
top,Female,Non-Hispanic White,College graduate or above,No,Yes,No,Insured,No,Public only
freq,6358,6217,2625,4878,952,10371,11007,3632,5048



Duplicated id count: <bound method Series.sum of 0        False
1        False
2        False
3        False
4        False
         ...  
11928    False
11929    False
11930    False
11931    False
11932    False
Name: id, Length: 11933, dtype: bool>


Missing Percentage by Variable


smoking_ever             0.90
glucose                  0.69
physical_activity_freq   0.69
a1c                      0.44
doctor_visit_12m         0.44
dbp                      0.37
sbp                      0.37
education                0.35
smoking_current          0.32
sleep_hours              0.30
bmi                      0.29
height_cm                0.29
weight_kg                0.27
income_poverty_ratio     0.17
diabetes_self_report     0.02
has_health_insurance     0.00
id                       0.00
age                      0.00
race                     0.00
gender                   0.00
insurance_type           0.00
dtype: float64

In [155]:
#Handle missing values 
#(fill categorical variables with 'Unknown' and numerical variables with the median by age x gender group➡️age➡️gobal).

cat_cols = [
    'gender','race','education',
    'smoking_current','smoking_ever',
    'diabetes_self_report',
    'has_health_insurance','doctor_visit_12m','insurance_type'
]

num_cols = [
    'age',                     
    'income_poverty_ratio',    
    'weight_kg', 'height_cm', 'bmi',   
    'sbp', 'dbp',              
    'a1c', 'glucose', 
    'physical_activity_freq', 'sleep_hours'     
]

for c in cat_cols:
    if c in nhanes_clean.columns:
        nhanes_clean[c] = nhanes_clean[c].astype('string').str.strip()
        nhanes_clean[c] = nhanes_clean[c].fillna("Unknown")

nhanes_clean['age_bin'] = pd.cut(
    nhanes_clean['age'],
    bins=[0,5,12,18,30,45,60,80],
    labels=['0-5','6-12','13-18','19-30','31-45','46-60','61-80']
)
for col in num_cols:
    if col not in nhanes_clean.columns or col == 'id':
        continue
    med_age_gender = nhanes_clean.groupby(['age_bin','gender'], observed=True)[col].transform('median')
    med_age = nhanes_clean.groupby('age_bin', observed=True)[col].transform('median')
    global_med = nhanes_clean[col].median()
    nhanes_clean[col] = nhanes_clean[col].fillna(med_age_gender).fillna(med_age).fillna(global_med)

print('Missing Percentage by Variable')
display(nhanes_clean.isna().mean().sort_values(ascending=False))


Missing Percentage by Variable


id                       0.00
age                      0.00
insurance_type           0.00
doctor_visit_12m         0.00
has_health_insurance     0.00
diabetes_self_report     0.00
sleep_hours              0.00
physical_activity_freq   0.00
smoking_ever             0.00
smoking_current          0.00
glucose                  0.00
a1c                      0.00
dbp                      0.00
sbp                      0.00
bmi                      0.00
height_cm                0.00
weight_kg                0.00
income_poverty_ratio     0.00
education                0.00
race                     0.00
gender                   0.00
age_bin                  0.00
dtype: float64

In [156]:
# Convert Object to Category datatype
cat_cols = [
    'gender','race','education',
    'smoking_current','smoking_ever',
    'diabetes_self_report',
    'has_health_insurance','doctor_visit_12m','insurance_type'
]
to_category = [c for c in cat_cols if c in nhanes_clean.columns]
nhanes_clean[to_category] = nhanes_clean[to_category].astype('category')

cat_cols = nhanes_clean.select_dtypes(include='category').columns.tolist()
print("Category columns:", cat_cols)

#Check for leading or trailing spaces
for c in nhanes_clean.select_dtypes(include='category').columns:
    s = nhanes_clean[c].astype('string')
    if s.str.match(r'^\s|.\s$').any():
        print(f"[hint] {c}: some values have leading/trailing spaces")

Category columns: ['gender', 'race', 'education', 'smoking_current', 'smoking_ever', 'diabetes_self_report', 'has_health_insurance', 'doctor_visit_12m', 'insurance_type', 'age_bin']


In [157]:
# Numeric datatype outlier check and handling with IQR

num_cols = [
    'age',                     
    'income_poverty_ratio',    
    'weight_kg', 'height_cm', 'bmi',   
    'sbp', 'dbp',              
    'a1c', 'glucose', 
    'physical_activity_freq', 'sleep_hours'     
]

for c in num_cols:
    q1, q3 = nhanes_clean[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    nhanes_clean[c + '_is_outlier'] = ~nhanes_clean[c].between(low, high)

outlier_rate = {
    c: float(nhanes_clean[c + '_is_outlier'].mean())
    for c in num_cols
}
outlier_table = pd.DataFrame.from_dict(outlier_rate, orient='index', columns=['Outlier (%)'])
outlier_table['Outlier (%)'] = outlier_table['Outlier (%)'] * 100
print(outlier_table.round(2))

# Define outliers based on the IQR and distributions 
#Replace outliers with medium by age x gender group➡️age➡️global

col = 'physical_activity_freq'
error_values = [9999, 8888, 7777]

nhanes_clean[col] = nhanes_clean[col].replace(error_values, np.nan)

if 'gender' in nhanes_clean.columns:
    med_age_gender = nhanes_clean.groupby(['age_bin','gender'], observed=True)[col].transform('median')
    nhanes_clean[col] = nhanes_clean[col].fillna(med_age_gender)

    med_age    = nhanes_clean.groupby('age_bin', observed=True)[col].transform('median')
    global_med = nhanes_clean[col].median()

    nhanes_clean[col] = nhanes_clean[col].fillna(med_age).fillna(global_med)
else:
    print(f"[skip] Column '{col}' not found.")

# Consistency validation
cols_to_check = ['weight_kg','height_cm','bmi','physical_activity_freq']
summary = nhanes_clean.groupby('age_bin',observed=True)[cols_to_check].agg(['count','mean','median','min','max']).round(1)
display(summary)

infant_group = nhanes_clean.loc[nhanes_clean['age'] <= 5, ['age'] + num_cols]

cols_to_remove = [c for c in nhanes_clean.columns if c.endswith('_is_outlier') or c == 'age_bin']
nhanes_clean.drop(columns=cols_to_remove, inplace=True)
display(nhanes_clean.describe(include='number'))
display(nhanes_clean.describe(include='category'))


nhanes_clean.to_csv("/Users/sherrywang/Desktop/Projects/Dataset/nhanes_clean.csv",
                    index=False)


                        Outlier (%)
age                            0.00
income_poverty_ratio           0.00
weight_kg                      1.65
height_cm                     11.26
bmi                            2.72
sbp                            2.41
dbp                            2.29
a1c                            6.46
glucose                        7.57
physical_activity_freq         5.35
sleep_hours                   31.56


weight_kg                           height_cm                       \
            count  mean median   min    max     count   mean median    min   
age_bin                                                                      
0-5          1237 15.30  15.40  2.70  42.30      1237 104.30 105.00  79.10   
6-12         1536 37.90  35.20 15.90 112.10      1536 138.50 137.60 106.60   
13-18        1191 67.90  64.40 24.70 172.90      1191 166.30 166.30 131.30   
19-30        1185 78.50  74.40 39.60 176.60      1185 169.00 168.00 146.10   
31-45        1734 84.80  80.80 32.60 248.20      1734 168.20 167.00 140.30   
46-60        1735 85.00  81.40 42.90 191.50      1735 167.10 166.30 133.00   
61-80        3315 80.40  77.60 27.90 192.10      3315 165.50 164.40 134.20   

                 bmi                          physical_activity_freq        \
           max count  mean median   min   max                  count  mean   
age_bin                                                                      
0-5     129.50  1237 16.30  16.20 12.50 30.80                   1237 45.00   
6-12    185.80  1536 19.10  18.00 12.40 43.60                   1536 45.00   
13-18   195.50  1191 24.40  22.80 12.90 66.60                   1191 60.70   
19-30   198.10  1185 27.40  26.40 14.90 61.50                   1185 58.10   
31-45   200.70  1734 29.90  28.80 14.10 74.80                   1734 50.40   
46-60   196.20  1735 30.50  29.60 15.20 69.90                   1735 54.50   
61-80   197.10  3315 29.30  28.70 11.10 68.50                   3315 49.50   

                             
        median   min    max  
age_bin                      
0-5      45.00 45.00  45.00  
6-12     45.00 45.00  45.00  
13-18    60.00  5.00 300.00  
19-30    45.00  4.00 540.00  
31-45    30.00  1.00 900.00  
46-60    45.00  1.00 600.00  
61-80    40.00  1.00 900.00

,id,age,income_poverty_ratio,weight_kg,height_cm,bmi,sbp,dbp,a1c,glucose,physical_activity_freq,sleep_hours
count,11933.00,11933.00,11933.00,11933.00,11933.00,11933.00,11933.00,11933.00,11933.00,11933.00,11933.00,11933.00
mean,136344.00,38.32,2.67,68.05,156.71,26.23,117.65,71.83,5.58,101.92,51.30,7.83
std,3444.90,25.60,1.54,29.33,22.25,7.61,15.83,10.08,0.81,18.88,33.65,1.36
min,130378.00,0.00,0.00,2.70,79.10,11.10,61.00,33.00,3.20,59.00,1.00,2.00
25%,133361.00,13.00,1.42,52.30,152.00,20.50,107.00,64.00,5.20,96.00,40.00,7.50
50%,136344.00,37.00,2.54,72.30,162.40,26.40,117.00,72.00,5.50,100.00,45.00,8.00
75%,139327.00,62.00,4.00,84.90,172.10,29.70,127.00,77.00,5.70,103.00,60.00,8.00
max,142310.00,80.00,5.00,248.20,200.70,74.80,232.00,142.00,17.10,561.00,900.00,14.00


,gender,race,education,smoking_current,smoking_ever,diabetes_self_report,has_health_insurance,doctor_visit_12m,insurance_type
count,11933,11933,11933,11933,11933,11933,11933,11933,11933
unique,2,6,6,5,3,5,5,5,5
top,Female,Non-Hispanic White,Unknown,No,Unknown,No,Insured,Unknown,Public only
freq,6358,6217,4150,4878,10743,10371,11007,5216,5048
